# The JSON Mode Revolution
Historically, getting JSON was a struggle of "only return JSON, no other text." Standard approach: Use native response_format: { type: "json_schema" } (OpenAI/Gemini) or tool-output schemas (Anthropic).

Benefit: 100% syntactical validity. The model literally cannot output a string that is not a valid JSON.
Behind the scenes: The serving engine masks the vocabulary at each step, ensuring only valid JSON characters (e.g., {, ", :, [) can be picked next.

# Direct Json through API

In [ ]:
from google import genai
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")

client = genai.Client(api_key="YOUR_API_KEY")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="My name is Ram and I am 21 years old.",
    config={
        "response_format": {
            "text": {
                "mime_type": "application/json",
                "schema": Person.model_json_schema(),
            }
        }
    },
)

person = Person.model_validate_json(response.text)
print(person)

## WIth Langchain which passes the same kwargs in backend

In [ ]:
# LangChain + Gemini

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

class Person(BaseModel):
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

structured_llm = llm.with_structured_output(Person)

result = structured_llm.invoke("My name is Ram and I am 21 years old.")
print(result)

## Giving the Response Structure in Prompt where the internal Masking is not supported

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

class Person(BaseModel):
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")

parser = PydanticOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_template(
    """
Extract the person info from the text.

{format_instructions}

Text:
{text}
"""
)

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

chain = prompt.partial(format_instructions=parser.get_format_instructions()) | llm | parser

result = chain.invoke({"text": "My name is Ram and I am 21 years old."})
print(result)

## Validation & Formatting Errors
Even with "JSON mode," the Logic inside the JSON might be wrong (e.g., a field is missing or a date is in the wrong format).

Recovery pattern:

- Validate output against Pydantic/Zod.
- If it fails, send the Traceback back to the model: "Error: Field 'age' must be an integer, got 'twenty'. Fix and re-generate."
- Most models fix the error on the first retry.

In [11]:
from openai import OpenAI
from pydantic import BaseModel, ValidationError
from dotenv import load_dotenv
import json
import os

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)


class Person(BaseModel):
    name: str
    age: int


def call_llm(prompt: str) -> str:

    response = client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content

    print("\n=== RAW MODEL OUTPUT ===")
    print(repr(content))
    print("========================\n")

    return content


def generate_person(text: str, max_retries: int = 2):

    prompt = f"""
Extract person information.

Return valid JSON.

Schema:

{{
    "name": string,
    "age": integer
}}

Text:
{text}
"""

    output = call_llm(prompt)

    for attempt in range(max_retries):

        try:

            data = json.loads(output)

            person = Person.model_validate(data)

            return person

        except (json.JSONDecodeError, ValidationError) as e:

            print(f"\nAttempt {attempt + 1} failed")
            print("Validation Error:", e)

            repair_prompt = f"""
The following JSON failed validation.

Validation Error:
{e}

Previous Output:
{output}

Return ONLY corrected JSON.
"""

            output = call_llm(repair_prompt)

    raise Exception("Failed after maximum retries")


if __name__ == "__main__":

    result = generate_person(
        "My name is Ram and my age is twenty one."
    )

    print("\nValidated Object:")
    print(result)

    print("\nAs Dict:")
    print(result.model_dump())


=== RAW MODEL OUTPUT ===
'{"name": "Ram", "age": 21}'


Validated Object:
name='Ram' age=21

As Dict:
{'name': 'Ram', 'age': 21}
